In [1]:
# preprocessing data & creating necessary documentation files

In [1]:
import pandas as pd 
import numpy as np
import pickle
import json
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [2]:
# generic config class
class FairnessConfig:
    """ 
    Configuration class for relevant fairness-monitoring tasks. User must specify
    datapath, target column(s), and protected attribute(s)

    args:
        data_path: path for specific dataset (.csv file)
        target: name of target column (ex: 'income')
        protected_attr: name of protected attribute (ex: 'sex')
        """
    def __init__(self, data_path, target, protected_attr, desired=None, privileged=None):
        self.data_path = data_path
        # load data from path
        self.df = pd.read_csv(data_path, na_values='?', header=None)

        self.target = target
        self.protected_attr = protected_attr
        
        if isinstance(self.df.columns[0], int):
            self.df = pd.read_csv(data_path, na_values='?', header=0)

        # auto-detect desired & privileged labels (binary 0/1, auto-selects second val alphabetically)
        if desired is None:
            vals = sorted(self.df[target].dropna().unique())
            self.desired = vals[-1]  # ex: 0/1 for >/< 50k
            print(f"auto-detected desired outcome: {self.desired}")
        else:
            self.desired = desired

        if privileged is None:
            vals = sorted(self.df[protected_attr].dropna().unique())
            self.privileged = vals[-1]  # ex: 'male', 'female'
            print(f"auto-detected privileged group: {self.privileged}")
        else:
            self.privileged = privileged

        # auto-detect feature cols, numeric vs categorical
        self.feature_cols = [c for c in self.df.columns if c not in [self.target, self.protected_attr]]
        self.numeric_cols = self.df[self.feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
        self.categorical_cols = self.df[self.feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

        print(f"\nauto-detected {len(self.feature_cols)} feature cols, {len(self.numeric_cols)} numeric cols, and {len(self.categorical_cols)} categorical cols")

    def to_dict(self):
        """ export to dictionary for .json """
        return {
            'data_path': self.data_path,
            'target': self.target,
            'protected_attr': self.protected_attr,
            'desired_label': self.desired,
            'privileged_label': self.privileged,
            'feature_cols': self.feature_cols,
            'numeric_cols': self.numeric_cols,
            'categorical_cols': self.categorical_cols }


In [3]:
# auto-preprocess
def auto_preprocess(config):
    """
    auto-preprocess any relevant dataset based on pre generated config

    returns:
        X,y, protected_attr, preprocessor
    """

    df = config.df.copy()
    
    # create binary target & protected attr (1 favorable, 0 unfavorable & 1 privileged, 0 unprivileged)

    y = (df[config.target] == config.desired).astype(int)
    protected_attr = (df[config.protected_attr] == config.privileged).astype(int)
    X = df[config.feature_cols]  # get features
    X.columns = [f'col_{i}' for i in range(len(X.columns))]

    # re-compute numeric and categorical after removing target/protected
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # create preprocessor pipeline
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)])

    X_processed = preprocessor.fit_transform(X)  # fit & transform

    num_cols = numeric_cols
    cat_cols = []
    if len(categorical_cols) > 0:
        cat_encoder = preprocessor.named_transformers_['cat']['encoder']
        cat_cols = cat_encoder.get_feature_names_out(categorical_cols).tolist()

    cols = num_cols + cat_cols
    X_df = pd.DataFrame(X_processed, columns=cols)

    print(f"input features: {X.shape[1]}")
    print(f"output features: {X_df.shape[1]}")
    print(f"target positive: {y.sum()} / {len(y)} ({y.sum()/len(y)*100:.1f}%")
    print(f"privileged group: {protected_attr.sum()} / {len(protected_attr)} ({protected_attr.sum()/len(protected_attr)*100:.1f}%)")
    return X_df, y, protected_attr, preprocessor

In [4]:
# preprocess
config = FairnessConfig(
    data_path='data/adult.data',
    target=14,
    protected_attr=9,  # auto-detect if not provided
    desired=' >50K',  # auto-detect if not provided
    privileged=' Male')

X, y, protected, preprocessor = auto_preprocess(config)
protected_raw = config.df[config.protected_attr].copy()


auto-detected 13 feature cols, 6 numeric cols, and 7 categorical cols
input features: 13
output features: 99
target positive: 7841 / 32561 (24.1%
privileged group: 21790 / 32561 (66.9%)


In [5]:
"""
VARIABLE KEYS: (for specific demo)
protected = protected_attribute binary (GENDER)
protected_raw = protecte_attr non-binary (SEX)
privileged_idx = male_idx (MALE) , binary_gender_col=1
unprivileged_idx = female_idx (FEMALE) , binary_gender_col=0
fair_priv_idx = fairness_mIdx (MALE FAIRNESS INDEX)
fair_unpriv_idx = fairness_fIDX (FEMALE FAIRNESS INDEX)
"""

'\nVARIABLE KEYS: (for specific demo)\nprotected = protected_attribute binary (GENDER)\nprotected_raw = protecte_attr non-binary (SEX)\nprivileged_idx = male_idx (MALE) , binary_gender_col=1\nunprivileged_idx = female_idx (FEMALE) , binary_gender_col=0\nfair_priv_idx = fairness_mIdx (MALE FAIRNESS INDEX)\nfair_unpriv_idx = fairness_fIDX (FEMALE FAIRNESS INDEX)\n'

In [6]:
'''# train-test split
X = df_encoded.drop(['target', 'protected', 'protected_raw', 'fnlwgt', 'education'], axis = 1)
y = df_encoded['target']
protected = df_encoded['protected'] # fairness
protected_raw = df_encoded['protected_raw'] # save for later'''

# separate out fairness set (20%)
X_temp, X_fair, y_temp, y_fair, protected_temp, protected_fair, protected_raw_temp, protected_raw_fair = train_test_split(
    X, y, protected, protected_raw, test_size = 0.20, random_state = 42, stratify = y)

# train/test from remaining 80% (60 : 20)
X_train, X_test, y_train, y_test, protected_train, protected_test, protected_raw_train, protected_raw_test = train_test_split(
    X_temp, y_temp, protected_temp, protected_raw_temp,  test_size = 0.25, random_state = 42, stratify = y_temp)

# balanced set for preservation
privileged_idx = protected_fair[protected_fair == 1].index
unprivileged_idx = protected_fair[protected_fair == 0].index

# sample equal num for each protected
min_size = min(len(privileged_idx), len(unprivileged_idx))
fair_priv_idx = np.random.choice(privileged_idx, min_size, replace = False)
fair_unpriv_idx = np.random.choice(unprivileged_idx, min_size, replace = False)
fairness_idx = np.concatenate([fair_priv_idx, fair_unpriv_idx])

# create fairness eval set
X_fair = X_fair.loc[fairness_idx]
y_fair = y_fair.loc[fairness_idx]
protected_fair = protected_fair.loc[fairness_idx]
protected_raw_fair = protected_raw_fair.loc[fairness_idx]

# shuffle data before creating set
shuffle_idx = np.random.RandomState(42).permutation(len(X_fair))
X_fair = X_fair.iloc[shuffle_idx].reset_index(drop=True)
y_fair = y_fair.iloc[shuffle_idx].reset_index(drop=True)
protected_fair = protected_fair.iloc[shuffle_idx].reset_index(drop=True)
protected_raw_fair = protected_raw_fair.iloc[shuffle_idx].reset_index(drop=True)


print(f"Training set size: {X_train.shape[0]} {X_train.shape[0] / len(X) * 100:.1f}%\n")
print(f"Test size set: {X_test.shape[0]}  {X_test.shape[0] / len(X) * 100:.1f}%\n")
print(f"Fairness eval set size: {X_fair.shape[0]}  {X_fair.shape[0] / len(X) * 100:.1f}%\n")
print(f"protected distributation in fairness set:\n {pd.Series(protected_fair).value_counts()}")

Training set size: 19536 60.0%

Test size set: 6512  20.0%

Fairness eval set size: 4316  13.3%

protected distributation in fairness set:
 9
0    2158
1    2158
Name: count, dtype: int64


In [7]:
# save processed data
processed_data = {
    'X_train' : X_train,
    'X_test': X_test,
    'X_fair' : X_fair,
    'y_train' : y_train,
    'y_test' : y_test,
    'y_fair' : y_fair,
    'protected_train' : protected_train,
    'protected_test' : protected_test,
    'protected_fair' : protected_fair,
    'protected_raw_train' : protected_raw_train,
    'protected_raw_test' : protected_raw_test,
    'protected_raw_fair' : protected_raw_fair,
    'preprocessor': preprocessor,
    'config': config}

# save pickle & export to json
with open('processed_data.pkl', 'wb') as f: pickle.dump(processed_data, f)
with open('dataset_config.json', 'w') as f: json.dump(config.to_dict(), f, indent=2)

# save csv
train_df = X_train.copy()
train_df['target'] = y_train
train_df['protected'] = protected_train
train_df.to_csv('train_processed.csv', index = False)

train_df = X_test.copy()
train_df['target'] = y_test
train_df['protected'] = protected_test
train_df.to_csv('test_processed.csv', index = False)

train_df = X_fair.copy()
train_df['target'] = y_fair
train_df['protected'] = protected_fair
train_df.to_csv('fairness_processed.csv', index = False)

print("processed_data.pkl & dataset_config.json - all processed datasets & config information")
print("train_processed.csv - training data")
print("test_processed.csv - test data")
print("fair_processed.csv - protected-based fairness evaluation set")

processed_data.pkl & dataset_config.json - all processed datasets & config information
train_processed.csv - training data
test_processed.csv - test data
fair_processed.csv - protected-based fairness evaluation set


In [10]:
# metadata abt preprosessing - for integrity
metadata = {
    'original_shape' : list(config.df.shape),
    'train_size' : int(X_train.shape[0]),
    'test_size' : int(X_test.shape[0]),
    'fairness_size' : int(X_fair.shape[0]),
    'target_distribution' : {
        'class_0': int((y == 0).sum()),  # from y
        'class_1': int((y == 1).sum())},
    'protected_distribution' : {
        'unprivileged': int((protected == 0).sum()),  # from protected
        'privileged': int((protected == 1).sum())},
    'protected_attribute' : 'protected',
    'favorable_outcome' : 1, # >50K 
    'unfavorable_outcome' : 0, # <50k
    'privileged_group' : 1, # male,
    'unprivileged_group' : 0, # female
    'preprocessing_date' : pd.Timestamp.now().strftime('%Y-%m-%d'),
    'scaling_method' : 'Auto-preprocessing pipeline',
    'data_leakage_prevented' : True,
    'num_features': X.shape[1],
    'original_col_num': len(config.feature_cols)}

with open('preprocessed_metadata.json', 'w') as f: json.dump(metadata, f, indent = 2)

In [11]:
# save original protected_attr (for bias simulation)
bias_sim_data = config.df[[config.protected_attr]].copy()
bias_sim_data.columns = ['protected_raw']
bias_sim_data.to_csv('og_bias_sim.csv', index=False)

In [12]:
# summary stats
print(f"\nOriginal dataset: {config.df.shape[0]} rows, {config.df.shape[1]} cols")
print(f"After preprocessing: {X.shape[1]} features")
print(f"Features for modeling: {X_train.shape[1]}\n")
print("target distribution:")
print(f"  Class 0 (unfavorable): {(y == 0).sum()} ({(y == 0).sum()/len(y) * 100:.1f}%)")
print(f"  Class 1 (favorable): {(y == 1).sum()} ({(y == 1).sum()/len(y) * 100:.1f}%)\n")
print("protected attribute distribution:")
print(f"  Unprivileged (0): {(protected == 0).sum()} ({(protected == 0).sum()/len(protected) * 100:.1f}%)")
print(f"  Privileged (1): {(protected == 1).sum()} ({(protected == 1).sum()/len(protected) * 100:.1f}%)\n")
print("data splits:")
print(f"  Training: {X_train.shape[0]} samples")
print(f"  Testing: {X_test.shape[0]} samples")
print(f"  Fairness monitoring: {X_fair.shape[0]} samples (balanced)")


Original dataset: 32561 rows, 15 cols
After preprocessing: 99 features
Features for modeling: 99

target distribution:
  Class 0 (unfavorable): 24720 (75.9%)
  Class 1 (favorable): 7841 (24.1%)

protected attribute distribution:
  Unprivileged (0): 10771 (33.1%)
  Privileged (1): 21790 (66.9%)

data splits:
  Training: 19536 samples
  Testing: 6512 samples
  Fairness monitoring: 4316 samples (balanced)
